# Lab 08-08 · Finite State Machines

Companion to chapter [08-08](../../chapters/08-08-finite-state-machines.md). Handout: [README.md](README.md).

Machines are stored as Python dictionaries, exactly as their state tables: a Moore machine maps each state to its pair of next states (for input 0 and 1) and has a set of states with output 1; a Mealy machine maps each state to a pair of (next state, output). Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import itertools
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MOORE101 = {0: (0, 1), 1: (2, 1), 2: (0, 3), 3: (2, 1)}            # S3 has output 1
MEALY101 = {0: ((0, 0), (1, 0)), 1: ((2, 0), (1, 0)), 2: ((0, 0), (1, 1))}
MOORE110 = {0: (0, 1), 1: (0, 2), 2: (3, 2), 3: (0, 1)}            # S3 has output 1


def run_moore(m, ones, xs, s=0):
    # output of each cycle 0 .. len(xs): the state during cycle t gives z_t
    out = []
    for x in xs:
        out.append(int(s in ones))
        s = m[s][x]
    out.append(int(s in ones))
    return out


def run_mealy(m, xs, s=0):
    out = []
    for x in xs:
        s, z = m[s][x][0], m[s][x][1]
        out.append(z)
    return out

## Part 1 · Moore and Mealy detectors against the specification

The specification of the 101 detector is a direct pattern match: the output for cycle $t$ (Mealy timing) is 1 if inputs $t-2$, $t-1$, $t$ were 1, 0, 1.

**🔮 Predict:** on random input, what fraction of cycles end a 101 pattern?

In [2]:
rng = np.random.default_rng(808)
xs = [int(b) for b in rng.integers(0, 2, 10_000)]
spec = [int(t >= 2 and xs[t - 2:t + 1] == [1, 0, 1]) for t in range(len(xs))]
z_mealy = run_mealy(MEALY101, xs)
z_moore = run_moore(MOORE101, {3}, xs)
assert z_mealy == spec, "Mealy detector disagrees with the specification"
assert z_moore[1:] == spec, "Moore detector, delayed one cycle, disagrees with the specification"
print("cycles:", len(xs), "  detections:", sum(spec), f"  fraction: {sum(spec) / len(xs):.4f}  (1/8 = 0.125)")
print("Mealy output leads Moore output by one cycle:", z_moore[1:] == z_mealy)

cycles: 10000   detections: 1223   fraction: 0.1223  (1/8 = 0.125)
Mealy output leads Moore output by one cycle: True


Each cycle ends a 101 pattern with probability $(1/2)^3 = 1/8$ when the bits are independent and fair; overlapping occurrences are counted, so this expectation holds exactly (by linearity of expectation, even though neighboring occurrences are not independent). The Moore output is the Mealy output delayed by one cycle, as chapter §3.2 derived.

## Part 2 · Partition refinement

The function below implements chapter §3.5 for Moore machines: start from the partition by outputs, and split blocks until two states share a block only if, for both inputs, their next states share a block. It returns the number of blocks at each step and the final class of each state.

In [3]:
def refine_moore(nxt, out):
    blk = dict(out)
    sizes = []
    while True:
        sizes.append(len(set(blk.values())))
        sig = {s: (blk[s], blk[nxt[s][0]], blk[nxt[s][1]]) for s in nxt}
        ids = {k: i for i, k in enumerate(sorted(set(sig.values()), key=repr))}
        new = {s: ids[sig[s]] for s in nxt}
        if len(set(new.values())) == len(set(blk.values())):
            return sizes, new
        blk = new


def outputs_from(nxt, out, s, xs):
    r = [out[s]]
    for x in xs:
        s = nxt[s][x]
        r.append(out[s])
    return r


def equivalent_after_merge(nxt, out, cls, depth=8):
    # build the merged machine and compare it with the original on every input sequence up to `depth`
    mini = {cls[s]: (cls[nxt[s][0]], cls[nxt[s][1]]) for s in nxt}
    mout = {cls[s]: out[s] for s in nxt}
    return all(outputs_from(nxt, out, s, xs) == outputs_from(mini, mout, cls[s], xs)
               for s in nxt for n in range(depth + 1) for xs in itertools.product((0, 1), repeat=n))


EX4 = {"A": ("B", "C"), "B": ("D", "E"), "C": ("F", "G"), "D": ("D", "A"), "E": ("D", "A"),
       "F": ("F", "A"), "G": ("F", "A")}
EX4OUT = {"A": 0, "B": 0, "C": 0, "D": 0, "E": 1, "F": 0, "G": 1}
sizes, cls = refine_moore(EX4, EX4OUT)
groups = sorted("".join(sorted(s for s in EX4 if cls[s] == c)) for c in set(cls.values()))
print("Example 08-08.4: blocks per step", sizes, " classes", groups)
assert sizes == [2, 3, 4] and groups == ["A", "BC", "DF", "EG"]
assert equivalent_after_merge(EX4, EX4OUT, cls)

Example 08-08.4: blocks per step [2, 3, 4]  classes ['A', 'BC', 'DF', 'EG']


Problem 08-08.7 is a Mealy machine: its initial partition groups states by their whole output row, and the refinement is otherwise the same.

In [4]:
def refine_mealy(m):
    blk = {s: (m[s][0][1], m[s][1][1]) for s in m}
    sizes = []
    while True:
        sizes.append(len(set(blk.values())))
        sig = {s: (blk[s], blk[m[s][0][0]], blk[m[s][1][0]]) for s in m}
        ids = {k: i for i, k in enumerate(sorted(set(sig.values()), key=repr))}
        new = {s: ids[sig[s]] for s in m}
        if len(set(new.values())) == len(set(blk.values())):
            return sizes, new
        blk = new


P7 = {"A": (("B", 0), ("C", 0)), "B": (("A", 0), ("D", 1)), "C": (("A", 0), ("E", 1)),
      "D": (("F", 0), ("D", 1)), "E": (("F", 0), ("E", 1)), "F": (("A", 0), ("C", 0))}
sizes7, cls7 = refine_mealy(P7)
print("Problem 08-08.7: blocks per step", sizes7,
      " classes", sorted("".join(sorted(s for s in P7 if cls7[s] == c)) for c in set(cls7.values())))
assert sizes7 == [2, 3, 4]

Problem 08-08.7: blocks per step [2, 3, 4]  classes ['A', 'BC', 'DE', 'F']


**Random machines.** Generate 300 random 6-state Moore machines (random next states and outputs), minimize each, and check every result by simulation.

**🔮 Predict:** what fraction of random machines can be reduced at all?

In [5]:
rng = np.random.default_rng(88)
reduced, final_sizes = 0, []
for trial in range(300):
    nxt = {s: (int(rng.integers(6)), int(rng.integers(6))) for s in range(6)}
    out = {s: int(rng.integers(2)) for s in range(6)}
    _, c = refine_moore(nxt, out)
    assert equivalent_after_merge(nxt, out, c, depth=7)
    k = len(set(c.values()))
    final_sizes.append(k)
    reduced += k < 6
print(f"{reduced} of 300 random 6-state machines could be reduced")
print("distribution of minimal sizes:", {k: final_sizes.count(k) for k in sorted(set(final_sizes))})

121 of 300 random 6-state machines could be reduced
distribution of minimal sizes: {1: 12, 2: 14, 3: 10, 4: 18, 5: 67, 6: 179}


Every minimized machine passed the equivalence check. With this seed, 121 of the 300 machines (about 40%) could be reduced: most by a single state (67 went from 6 to 5), and 12 collapsed to one state, which happens exactly when all six states have the same output (probability $2/2^6$, so about 9 expected in 300). Two states merge whenever no input sequence can separate them, which for random tables usually means a group of states with equal outputs that lead only into each other. Machines written by people are reducible for a different reason: they often contain states written separately for clarity that behave identically.

## Part 3 · How much does the state encoding matter? (the chapter's guided exercise)

The Moore 110 detector has four states, so there are $4! = 24$ ways to assign the codes 00, 01, 10, 11. For each, the cell below writes the next-state bits $q_1^{+}, q_0^{+}$ as functions of $(q_1, q_0, x)$ and the output $z$ as a function of $(q_1, q_0)$, and finds each function's **exact** minimum sum of products by trying every set of product terms.

**🔮 Predict:** how far apart are the best and the worst encodings, in total literals?

In [6]:
def covers(cube, m, n):
    return all(c is None or c == (m >> (n - 1 - i)) & 1 for i, c in enumerate(cube))


def minlits(on, dc=(), n=3):
    on, dc = set(on), set(dc)
    if not on:
        return 0
    imps = [c for c in itertools.product((0, 1, None), repeat=n)
            if all(m in on or m in dc for m in range(2 ** n) if covers(c, m, n))]
    best = None
    for k in range(1, len(on) + 1):
        for combo in itertools.combinations(imps, k):
            if all(any(covers(c, m, n) for c in combo) for m in on):
                lits = sum(sum(v is not None for v in c) for c in combo)
                best = lits if best is None else min(best, lits)
        if best is not None and k >= 2:
            return best
    return best


def encoding_cost(m, ones, code):
    q1, q0 = [], []
    for s, nx in m.items():
        for x in (0, 1):
            idx = (code[s] << 1) | x
            if (code[nx[x]] >> 1) & 1:
                q1.append(idx)
            if code[nx[x]] & 1:
                q0.append(idx)
    return minlits(q1), minlits(q0), minlits([code[s] for s in ones], (), 2)


results = []
for perm in itertools.permutations(range(4)):
    code = dict(enumerate(perm))
    results.append((sum(encoding_cost(MOORE110, {3}, code)), code, encoding_cost(MOORE110, {3}, code)))
results.sort(key=lambda r: r[0])
totals = [r[0] for r in results]
print("total literals of the 24 encodings:", totals)
for tot, code, parts in results:
    if tot == totals[0]:
        print("best:", {f"S{s}": format(c, "02b") for s, c in code.items()}, " (q1+, q0+, z) literals:", parts)
straight = sum(encoding_cost(MOORE110, {3}, {0: 0, 1: 1, 2: 2, 3: 3}))
print("straightforward S0=00, S1=01, S2=10, S3=11:", straight)
assert (len(totals), totals[0], totals[-1], totals.count(totals[0]), straight) == (24, 6, 17, 4, 16)

total literals of the 24 encodings: [6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 9, 9, 9, 9, 16, 16, 16, 16, 17, 17, 17, 17]
best: {'S0': '01', 'S1': '11', 'S2': '10', 'S3': '00'}  (q1+, q0+, z) literals: (1, 3, 2)
best: {'S0': '10', 'S1': '11', 'S2': '01', 'S3': '00'}  (q1+, q0+, z) literals: (3, 1, 2)
best: {'S0': '11', 'S1': '01', 'S2': '00', 'S3': '10'}  (q1+, q0+, z) literals: (1, 3, 2)
best: {'S0': '11', 'S1': '10', 'S2': '00', 'S3': '01'}  (q1+, q0+, z) literals: (3, 1, 2)
straightforward S0=00, S1=01, S2=10, S3=11: 16


The spread is almost a factor of three, for a four-state machine. The four best encodings share a structure: one bit records the last input (the states entered on input 1, "seen 1" and "seen 11", share its value, so its next-state function is just $x$ or $\overline{x}$), and the other bit separates {S0, S1} from {S2, S3}, with {S0, S1} coded as 1. Encoding B of Problem 08-08.5 has the same structure with that second bit's polarity reversed, and costs one literal more (7). The straightforward assignment, numbering the states in the order they were named, is nearly the worst. Synthesis tools explore encodings for exactly this reason, and for large machines they cannot try them all.

In [7]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
vals, counts = np.unique(totals, return_counts=True)
ax.bar(vals, counts, color="#0072B2")
ax.set_xlabel("total literals of minimized logic")
ax.set_ylabel("number of encodings")
ax.set_title("The 24 encodings of the Moore 110 detector")
ax.set_xticks(range(5, 19))
plt.show()

## Part 4 · Illegal codes and upset detection

With binary codes $0 \ldots N-1$ in $n = \lceil \log_2 N \rceil$ bits, a single flipped bit is detectable only if it lands on an unused code. One-hot codes are at distance 2 from each other, so every single flip is detectable.

In [8]:
def flip_detect(codes, nbits):
    legal = set(codes)
    total = len(codes) * nbits
    bad = sum((c ^ (1 << b)) not in legal for c in codes for b in range(nbits))
    return total, bad


print(" N  bits  illegal  detectable flips  (one-hot)")
rows = []
for N in range(2, 17):
    nb_ = max(1, math.ceil(math.log2(N)))
    tot, bad = flip_detect(list(range(N)), nb_)
    rows.append((N, 100 * bad / tot))
    print(f"{N:2d}  {nb_:4d}  {2 ** nb_ - N:7d}  {bad:3d} of {tot:3d} = {100 * bad / tot:5.1f}%   (100%)")
t12, b12 = flip_detect(list(range(12)), 4)
assert (t12, b12) == (48, 8)

 N  bits  illegal  detectable flips  (one-hot)
 2     1        0    0 of   2 =   0.0%   (100%)
 3     2        1    2 of   6 =  33.3%   (100%)
 4     2        0    0 of   8 =   0.0%   (100%)
 5     3        3    5 of  15 =  33.3%   (100%)
 6     3        2    4 of  18 =  22.2%   (100%)
 7     3        1    3 of  21 =  14.3%   (100%)
 8     3        0    0 of  24 =   0.0%   (100%)
 9     4        7   10 of  36 =  27.8%   (100%)
10     4        6   10 of  40 =  25.0%   (100%)
11     4        5   10 of  44 =  22.7%   (100%)
12     4        4    8 of  48 =  16.7%   (100%)
13     4        3    8 of  52 =  15.4%   (100%)
14     4        2    6 of  56 =  10.7%   (100%)
15     4        1    4 of  60 =   6.7%   (100%)
16     4        0    0 of  64 =   0.0%   (100%)


Binary encoding detects nothing when $N$ is a power of 2 (there are no unused codes) and at most about a third of single flips otherwise. That is why safe-FSM options in synthesis tools re-encode the machine with distance-2 or distance-3 codes rather than relying on the `default` branch of a binary machine.

**Verilog.** Now run `fsm_tb.v` (`python tools/run_hdl.py` from the repository root). Its detection counts come from Icarus's own random generator, so they differ from Part 1, but the fraction should again be close to 1/8.